# $\Pi_0^1$ projector — numerical validation

This notebook is **separate from the clean application demo**.  It tests the packaged implementation numerically without changing the public API.

It includes the checks already established during development:

1. exact reproduction of a lowest-order Nédélec field;
2. $L^2$ and curl errors for a smooth quadratic field;
3. first-order $L^2$ and curl convergence for a field with nonconstant curl;
4. the commuting check
   $$\nabla\Pi_0^0 u = \Pi_0^1(\nabla u).$$

The user-facing notebook remains just `P = Pi1Projector(mesh)` and `Pi1_u = P(u)`.


In [ ]:
# Install NGSolve in Colab if needed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("ngsolve") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ngsolve==6.2.2606", "webgui_jupyter_widgets"])
print("NGSolve is available.")


In [ ]:
# Load the package module.
import os
if not os.path.exists("pi1projector.py"):
    from google.colab import files
    print("Please upload pi1projector.py")
    files.upload()

import numpy as np
import pi1projector as p1
from pi1projector import Pi1Projector
from ngsolve import *


## Validation mesh family

For the convergence study we use the deterministic uniform tetrahedral cube family from `pi1projector.py`. The mesh-generator implementation is kept in the external module, so the notebook only contains the actual tests. The **application demo** separately uses a nontrivial Netgen/OCC Fichera mesh.


In [ ]:
# Validation mesh helpers live in pi1projector.py.
# The notebook only calls them; their implementation stays out of sight.
print("validation helpers loaded from pi1projector.py")


## 1. Exact lowest-order Nédélec reproduction

A field of the form $a+b\times x$ belongs to the lowest-order Nédélec space.  A projection should reproduce it to roundoff.


In [ ]:
mesh_data = p1.structured_unit_cube_six_tets(3)
mesh = p1.ngmesh_from_data(mesh_data)
P = Pi1Projector(mesh)

X = CF((x,y,z))
a = CF((1.0, 2.0, 3.0))
b = CF((0.3, -0.4, 0.5))
u_ned = a + Cross(b, X)
Pi1_ned = P(u_ned)

err = sqrt(Integrate(InnerProduct(Pi1_ned-u_ned, Pi1_ned-u_ned), mesh, order=8))
nrm = sqrt(Integrate(InnerProduct(u_ned,u_ned), mesh, order=8))
print("exact Nedelec reproduction L2 error     =", err)
print("exact Nedelec reproduction relative err =", err/nrm)


## 2. Single-mesh $L^2$ and curl error

For
$$u=(x^2+y,\;y^2+z,\;z^2+x),$$
we have $\operatorname{curl}u=(-1,-1,-1)$.  On the $n=6$ mesh the function has a genuine lowest-order approximation error, while the constant curl is reproduced to roundoff.


In [ ]:
mesh_data = p1.structured_unit_cube_six_tets(6)
mesh = p1.ngmesh_from_data(mesh_data)
P = Pi1Projector(mesh)

u_quad = CF((x*x+y, y*y+z, z*z+x))
curl_quad = CF((-1.0,-1.0,-1.0))
Pi1_quad = P(u_quad)

l2_err = sqrt(Integrate(InnerProduct(Pi1_quad-u_quad, Pi1_quad-u_quad), mesh, order=8))
l2_norm = sqrt(Integrate(InnerProduct(u_quad,u_quad), mesh, order=8))
curl_err = sqrt(Integrate(InnerProduct(curl(Pi1_quad)-curl_quad, curl(Pi1_quad)-curl_quad), mesh, order=8))
curl_norm = sqrt(Integrate(InnerProduct(curl_quad,curl_quad), mesh, order=8))

print("FUNCTION")
print("L2 error          =", l2_err)
print("relative L2 error =", l2_err/l2_norm)
print("\nCURL")
print("curl L2 error          =", curl_err)
print("relative curl L2 error =", curl_err/curl_norm)


## 3. $L^2$ and curl convergence for nonconstant curl

Take
$$u=(0,0,xy),\qquad \operatorname{curl}u=(x,-y,0).$$
Both the $L^2$ error and curl error should approach first order for the lowest-order target space.


In [ ]:
u_nc = CF((0.0, 0.0, x*y))
curl_nc = CF((x, -y, 0.0))

old_h = old_l2 = old_curl = None
print(f"{'n':>3} {'tets':>8} {'h':>12} {'L2 err':>14} {'L2 rate':>10} {'curl err':>14} {'curl rate':>10}")
print("-"*90)

for n in (2,3,4,5,6):
    md = p1.structured_unit_cube_six_tets(n)
    mesh = p1.ngmesh_from_data(md)
    P = Pi1Projector(mesh)
    w = P(u_nc, name=f"Pi1_u_n{n}")

    e0 = sqrt(Integrate(InnerProduct(w-u_nc,w-u_nc), mesh, order=8))
    ec = sqrt(Integrate(InnerProduct(curl(w)-curl_nc,curl(w)-curl_nc), mesh, order=8))
    h = p1.tetrahedral_hmax(md)

    r0 = np.nan if old_h is None else np.log(old_l2/e0)/np.log(old_h/h)
    rc = np.nan if old_h is None else np.log(old_curl/ec)/np.log(old_h/h)
    r0s = '---' if np.isnan(r0) else f'{r0:.6f}'
    rcs = '---' if np.isnan(rc) else f'{rc:.6f}'

    print(f"{n:3d} {md.tets.shape[0]:8d} {h:12.6e} {e0:14.8e} {r0s:>10} {ec:14.8e} {rcs:>10}")
    old_h, old_l2, old_curl = h, e0, ec


## 4. Direct gradient commuting check

This validation uses the full internal validation builder and the module helper `apply_pi0_for_validation`. The helper implementation remains in `pi1projector.py`, keeping this notebook focused on the test itself. It checks
$$\nabla\Pi_0^0u=\Pi_0^1(\nabla u).$$


In [ ]:
mesh_data = p1.structured_unit_cube_six_tets(4)
result = p1.build_faithful_pi1_matrix(
    mesh_data, tolerance=5e-8, retain_local_weights=False, verbose=False
)
mesh = result.coarse_patch.mesh

u_scalar = x*x + x*y + y*z + z*z
grad_u = CF((u_scalar.Diff(x), u_scalar.Diff(y), u_scalar.Diff(z)))

Pi0_u, c0 = p1.apply_pi0_for_validation(
    result, u_scalar, name="Pi0_u", integration_order=8
)
Pi1_grad_u, c1 = p1.apply_pi1_to_l2_field(
    result, grad_u, name="Pi1_grad_u", integration_order=8
)

comm_l2 = sqrt(Integrate(
    InnerProduct(grad(Pi0_u)-Pi1_grad_u, grad(Pi0_u)-Pi1_grad_u),
    mesh, order=8
))
comm_coeff = np.max(np.abs(result.G0 @ c0 - c1))

print("||grad Pi0(u) - Pi1(grad u)||_L2 =", comm_l2)
print("max HCurl coefficient error           =", comm_coeff)


## Expected behavior

The previously observed results were:

- Nédélec-range reproduction: roundoff;
- quadratic $n=6$ relative $L^2$ error about $0.1115$ and curl error at roundoff;
- nonconstant-curl $L^2$ and curl rates approaching $1$;
- gradient commuting error about $10^{-14}$.

This notebook exists to keep that evidence with the package while leaving the application notebook clean.
